In [ ]:
from operator import itemgetter
from pathlib import Path

from mmaction.apis import init_recognizer, inference_recognizer

# This notebook lives in notebooks/, one level below the repository root.
ROOT = Path.cwd().parent

In [ ]:
# Choose to use a config and initialize the recognizer
config_path = ROOT / "configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py"
checkpoint_path = ROOT / "checkpoints/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb_20220906-cd10898e.pth"

In [ ]:
# build the model from a config file and a checkpoint file
model = init_recognizer(config_path, str(checkpoint_path), device='cuda:0')

In [ ]:
# test a single video and show the result:
video = ROOT / "videos/demo.mp4"
label = ROOT / "tools/data/kinetics/label_map_k400.txt"
pred_result = inference_recognizer(model, str(video))

#print(pred_result)
pred_scores = pred_result.pred_score.tolist()
print(len(pred_scores))

score_tuples = tuple(zip(range(len(pred_scores)), pred_scores))
score_sorted = sorted(score_tuples, key=itemgetter(1), reverse=True)
top5_label = score_sorted[:5]

labels = [line.strip() for line in label.read_text().splitlines()]
results = [(labels[k[0]], k[1]) for k in top5_label]

In [ ]:
# show the results
for result in results:
    print(f'{result[0]}: ', result[1])

In [ ]:
# render the top-3 predictions onto the video frames and display it inline (no file left on disk)
import tempfile

from IPython.display import Video, display
from mmengine.structures import LabelData

from mmaction.visualization import ActionVisualizer

# `inference_recognizer` only sets `pred_score` on the result; the visualizer
# looks for a separate `pred_labels` field (top predicted class indices + the
# score vector) to draw the prediction text, so build that field ourselves.
# The visualizer's own text formatting already rounds scores to 2dp.
top3_idx = pred_result.pred_score.topk(3).indices
pred_result.pred_labels = LabelData(item=top3_idx, score=pred_result.pred_score)

visualizer = ActionVisualizer()
visualizer.dataset_meta = dict(classes=labels)

with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir) / "demo_out.mp4"
    visualizer.add_datasample(
        tmp_path.name,
        str(video),
        pred_result,
        draw_pred=True,
        draw_gt=False,
        text_cfg=dict(colors='white'),
        fps=30,
        out_type='video',
        out_path=str(tmp_path),
    )
    # display() reads and embeds the file now, while it still exists;
    # the temp file is deleted once this block exits.
    display(Video(str(tmp_path), embed=True))